# 🛡️ DeepGuard — AI Model Training Notebook
## IBM AML Dataset — Isolation Forest + Deep Autoencoder + SHAP
### Iqra University FYP — Batch 2023

**Instructions:**
1. Upload IBM AML dataset CSV to Colab Files panel
2. Run all cells top to bottom
3. Download the exported models at the end
4. Place models in `fastapi/models/` folder

**Dataset:** IBM Transactions for Anti-Money Laundering (HI_Small_Trans.csv)
**Kaggle:** https://www.kaggle.com/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml

## 📦 Step 1: Install Dependencies

In [ ]:
# Install all required packages
!pip install -q kaggle pandas numpy scikit-learn tensorflow shap matplotlib seaborn joblib plotly
print('✅ All packages installed successfully')

## 📥 Step 2: Download Dataset from Kaggle

In [ ]:
# OPTION A: Upload kaggle.json for automatic download
# Go to Kaggle > Account > Create API Token > Upload kaggle.json here

import os
from google.colab import files

print('Choose how to load dataset:')
print('OPTION A: Upload kaggle.json then run the kaggle download cell')
print('OPTION B: Manually upload HI_Small_Trans.csv to Colab Files panel')
print()
print('For OPTION B - just upload the CSV and skip to Step 3')

In [ ]:
# OPTION A ONLY - Skip if using manual upload
# Upload your kaggle.json first
try:
    uploaded = files.upload()  # Upload kaggle.json
    !mkdir -p ~/.kaggle
    !cp kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json
    !kaggle datasets download -d ealtman2019/ibm-transactions-for-anti-money-laundering-aml
    !unzip -q ibm-transactions-for-anti-money-laundering-aml.zip
    print('✅ Dataset downloaded from Kaggle')
except:
    print('⚠️ Skipped Kaggle download - use manual upload')

## 🔧 Step 3: Import Libraries & Load Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
import seaborn as sns
import warnings
import joblib
import json
import os
warnings.filterwarnings('ignore')

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, classification_report,
    confusion_matrix, f1_score,
    precision_score, recall_score
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

import shap

print(f'✅ TensorFlow version: {tf.__version__}')
print(f'✅ GPU available: {len(tf.config.list_physical_devices("GPU")) > 0}')
print('✅ All libraries loaded')

In [ ]:
# ============================================================
# LOAD DATASET
# Update filename if yours is different
# ============================================================

# Try common filenames for IBM AML dataset
possible_files = [
    'HI_Small_Trans.csv',
    'HI-SmallTrans.csv',
    'LI_Small_Trans.csv',
    'HI_Small_Trans.csv'
]

df = None
for fname in possible_files:
    if os.path.exists(fname):
        df = pd.read_csv(fname)
        print(f'✅ Loaded: {fname}')
        break

if df is None:
    # Try loading any CSV file uploaded
    csv_files = [f for f in os.listdir('.') if f.endswith('.csv')]
    if csv_files:
        df = pd.read_csv(csv_files[0])
        print(f'✅ Loaded: {csv_files[0]}')
    else:
        raise FileNotFoundError('❌ No CSV found. Please upload the IBM AML dataset CSV file.')

print(f'📊 Shape: {df.shape}')
print(f'📊 Columns: {list(df.columns)}')
df.head()

## 📊 Step 4: Exploratory Data Analysis

In [ ]:
print('=' * 60)
print('DATASET OVERVIEW')
print('=' * 60)
print(f'Total Transactions:    {len(df):,}')
print(f'Total Columns:         {len(df.columns)}')
print(f'Memory Usage:          {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB')
print()

# Check for Is Laundering column (handle different naming)
label_col = None
for col in df.columns:
    if 'launder' in col.lower() or 'fraud' in col.lower() or 'label' in col.lower():
        label_col = col
        break

if label_col:
    laundering_count = df[label_col].sum()
    total = len(df)
    print(f'Laundering Transactions: {laundering_count:,} ({laundering_count/total*100:.3f}%)')
    print(f'Legitimate Transactions: {total - laundering_count:,} ({(total-laundering_count)/total*100:.3f}%)')
    print(f'Laundering Ratio:        1 in every {int(total/laundering_count):,} transactions')
else:
    print('⚠️ No label column found - running in unsupervised mode only')

print()
print('DATA TYPES:')
print(df.dtypes)
print()
print('MISSING VALUES:')
print(df.isnull().sum())

In [ ]:
# Visualize class distribution
if label_col:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle('DeepGuard — IBM AML Dataset Analysis', fontsize=14, fontweight='bold')

    # Class distribution
    counts = df[label_col].value_counts()
    axes[0].bar(['Legitimate', 'Laundering'], counts.values,
                color=['#2ecc71', '#e74c3c'], edgecolor='black')
    axes[0].set_title('Transaction Distribution')
    axes[0].set_ylabel('Count')
    for i, v in enumerate(counts.values):
        axes[0].text(i, v + 0.01*v, f'{v:,}', ha='center', fontweight='bold')

    # Amount distribution
    if 'Amount Paid' in df.columns:
        axes[1].hist(df[df[label_col]==0]['Amount Paid'].clip(upper=df['Amount Paid'].quantile(0.99)),
                    bins=50, alpha=0.6, color='#2ecc71', label='Legitimate')
        axes[1].hist(df[df[label_col]==1]['Amount Paid'].clip(upper=df['Amount Paid'].quantile(0.99)),
                    bins=50, alpha=0.6, color='#e74c3c', label='Laundering')
        axes[1].set_title('Amount Paid Distribution')
        axes[1].set_xlabel('Amount')
        axes[1].legend()

    # Payment Format distribution
    if 'Payment Format' in df.columns:
        pf_counts = df['Payment Format'].value_counts()
        axes[2].barh(pf_counts.index, pf_counts.values, color='#3498db')
        axes[2].set_title('Payment Format Distribution')
        axes[2].set_xlabel('Count')

    plt.tight_layout()
    plt.savefig('eda_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('✅ EDA plot saved')

## 🔧 Step 5: Data Preprocessing

In [ ]:
print('Starting preprocessing pipeline...')
df_processed = df.copy()

# ============================================================
# 1. STANDARDIZE COLUMN NAMES
# ============================================================
# Handle Account.1 naming from IBM dataset
col_mapping = {}
cols = list(df_processed.columns)
account_cols = [c for c in cols if 'account' in c.lower()]
if len(account_cols) >= 2:
    col_mapping[account_cols[0]] = 'From Account'
    col_mapping[account_cols[1]] = 'To Account'

if col_mapping:
    df_processed = df_processed.rename(columns=col_mapping)

# Standardize label column
if label_col and label_col != 'Is Laundering':
    df_processed = df_processed.rename(columns={label_col: 'Is Laundering'})
    label_col = 'Is Laundering'

print(f'✅ Columns after standardization: {list(df_processed.columns)}')

# ============================================================
# 2. DROP MISSING VALUES
# ============================================================
before = len(df_processed)
df_processed = df_processed.dropna()
after = len(df_processed)
print(f'✅ Dropped {before - after:,} rows with missing values')

# ============================================================
# 3. TIMESTAMP FEATURE ENGINEERING
# ============================================================
if 'Timestamp' in df_processed.columns:
    try:
        df_processed['Timestamp'] = pd.to_datetime(df_processed['Timestamp'])
        df_processed['Hour'] = df_processed['Timestamp'].dt.hour
        df_processed['DayOfWeek'] = df_processed['Timestamp'].dt.dayofweek
        df_processed['DayOfMonth'] = df_processed['Timestamp'].dt.day
        df_processed['Month'] = df_processed['Timestamp'].dt.month
        # Is it a weekend?
        df_processed['IsWeekend'] = (df_processed['DayOfWeek'] >= 5).astype(int)
        # Is it off-hours (before 8am or after 8pm)?
        df_processed['IsOffHours'] = ((df_processed['Hour'] < 8) | (df_processed['Hour'] > 20)).astype(int)
        print('✅ Timestamp features extracted: Hour, DayOfWeek, DayOfMonth, Month, IsWeekend, IsOffHours')
    except Exception as e:
        print(f'⚠️ Timestamp parsing failed: {e}')

# ============================================================
# 4. AMOUNT DIFFERENCE FEATURE
# ============================================================
if 'Amount Paid' in df_processed.columns and 'Amount Received' in df_processed.columns:
    df_processed['Amount Difference'] = abs(
        df_processed['Amount Paid'] - df_processed['Amount Received']
    )
    df_processed['Amount Ratio'] = df_processed['Amount Received'] / (
        df_processed['Amount Paid'] + 1e-8
    )
    print('✅ Amount features added: Amount Difference, Amount Ratio')

# ============================================================
# 5. ONE-HOT ENCODING — Payment Format
# ============================================================
categorical_cols = ['Payment Format', 'Payment Currency', 'Receiving Currency']
existing_cat = [c for c in categorical_cols if c in df_processed.columns]

for col in existing_cat:
    dummies = pd.get_dummies(df_processed[col], prefix=col)
    df_processed = pd.concat([df_processed, dummies], axis=1)
    df_processed.drop(columns=[col], inplace=True)
    print(f'✅ One-Hot Encoded: {col}')

print(f'\n✅ Preprocessing complete. Shape: {df_processed.shape}')

In [ ]:
# ============================================================
# BUILD FEATURE MATRIX
# ============================================================

# Columns to drop from features
drop_cols = ['Timestamp', 'From Account', 'To Account',
             'From Bank', 'To Bank', 'Is Laundering']
drop_cols = [c for c in drop_cols if c in df_processed.columns]

# Feature matrix
X = df_processed.drop(columns=drop_cols)

# Labels (for evaluation only - NOT used in training)
y = df_processed['Is Laundering'].values if 'Is Laundering' in df_processed.columns else None

# Keep only numeric columns
X = X.select_dtypes(include=[np.number])

print(f'✅ Feature matrix shape: {X.shape}')
print(f'✅ Features used ({len(X.columns)}):')
for i, col in enumerate(X.columns):
    print(f'   {i+1:2d}. {col}')

# Save feature names for FastAPI
feature_names = list(X.columns)
with open('feature_names.json', 'w') as f:
    json.dump(feature_names, f)
print(f'\n✅ Feature names saved to feature_names.json')

In [ ]:
# ============================================================
# MIN-MAX NORMALIZATION
# ============================================================

scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

print(f'✅ Min-Max Normalization applied')
print(f'   Min values range: {X_scaled.min().min():.4f}')
print(f'   Max values range: {X_scaled.max().max():.4f}')

# Save scaler
joblib.dump(scaler, 'scaler.pkl')
print('✅ Scaler saved to scaler.pkl')

# Train/Test Split (60/20/20 as recommended by IBM)
X_train, X_temp, y_train, y_temp = train_test_split(
    X_scaled, y, test_size=0.40, random_state=42, stratify=y if y is not None else None
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp if y_temp is not None else None
)

print(f'\n✅ Data Split (60/20/20):')
print(f'   Training:   {len(X_train):,} samples')
print(f'   Validation: {len(X_val):,} samples')
print(f'   Testing:    {len(X_test):,} samples')

# For autoencoder - train on NORMAL transactions only (unsupervised)
if y_train is not None:
    X_train_normal = X_train[y_train == 0]
    print(f'   Normal (for AE training): {len(X_train_normal):,} samples')
else:
    X_train_normal = X_train
    print('   Training on all samples (no labels available)')

## 🌲 Step 6: Train Isolation Forest

In [ ]:
print('=' * 60)
print('TRAINING ISOLATION FOREST')
print('=' * 60)

# Isolation Forest Configuration
# contamination = approximate fraud ratio in dataset
contamination_rate = 0.001  # 1 in 1000 ≈ IBM AML ratio

iso_forest = IsolationForest(
    n_estimators=100,        # Number of trees
    contamination=contamination_rate,
    max_samples='auto',
    max_features=1.0,
    bootstrap=False,
    n_jobs=-1,               # Use all CPU cores
    random_state=42,
    verbose=0
)

print(f'Config: n_estimators=100, contamination={contamination_rate}')
print('Training... (this may take a few minutes)')

iso_forest.fit(X_train)
print('✅ Isolation Forest trained successfully')

# Get scores on test set
# score_samples returns negative — more negative = more anomalous
if_scores_raw = iso_forest.score_samples(X_test)
if_predictions = iso_forest.predict(X_test)  # -1=anomaly, 1=normal

# Convert to 0-100 risk score (higher = more suspicious)
if_scores_norm = 1 - (if_scores_raw - if_scores_raw.min()) / (if_scores_raw.max() - if_scores_raw.min())
if_scores_pct = (if_scores_norm * 100).round(2)

# Convert predictions: -1 → 1 (fraud), 1 → 0 (normal)
if_binary = np.where(if_predictions == -1, 1, 0)

print(f'\n📊 Isolation Forest Results on Test Set:')
print(f'   Anomalies detected: {if_binary.sum():,} / {len(if_binary):,}')

if y_test is not None:
    print(f'\n📊 Performance Metrics:')
    print(f'   ROC-AUC:   {roc_auc_score(y_test, if_scores_norm):.4f}')
    print(f'   F1 Score:  {f1_score(y_test, if_binary, zero_division=0):.4f}')
    print(f'   Precision: {precision_score(y_test, if_binary, zero_division=0):.4f}')
    print(f'   Recall:    {recall_score(y_test, if_binary, zero_division=0):.4f}')

# Save model
joblib.dump(iso_forest, 'isolation_forest.pkl')
print('\n✅ Isolation Forest saved to isolation_forest.pkl')

## 🧠 Step 7: Train Deep Autoencoder

In [ ]:
print('=' * 60)
print('TRAINING DEEP AUTOENCODER')
print('=' * 60)

input_dim = X_train_normal.shape[1]
print(f'Input dimensions: {input_dim}')

# ============================================================
# AUTOENCODER ARCHITECTURE
# Input → Dense 32 → Dense 16 → Bottleneck 8 → Dense 16 → Dense 32 → Output
# ============================================================

def build_autoencoder(input_dim):
    inputs = keras.Input(shape=(input_dim,), name='input')

    # ENCODER
    x = layers.Dense(32, activation='relu', name='encoder_1')(inputs)
    x = layers.BatchNormalization(name='bn_1')(x)
    x = layers.Dropout(0.2, name='dropout_1')(x)
    x = layers.Dense(16, activation='relu', name='encoder_2')(x)
    x = layers.BatchNormalization(name='bn_2')(x)
    encoded = layers.Dense(8, activation='relu', name='bottleneck')(x)

    # DECODER
    x = layers.Dense(16, activation='relu', name='decoder_1')(encoded)
    x = layers.BatchNormalization(name='bn_3')(x)
    x = layers.Dense(32, activation='relu', name='decoder_2')(x)
    x = layers.BatchNormalization(name='bn_4')(x)
    outputs = layers.Dense(input_dim, activation='sigmoid', name='output')(x)

    autoencoder = Model(inputs, outputs, name='DeepGuard_Autoencoder')
    return autoencoder

autoencoder = build_autoencoder(input_dim)
autoencoder.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='mse'
)

autoencoder.summary()

# Callbacks
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-6,
        verbose=1
    )
]

print('\nTraining Autoencoder on NORMAL transactions only...')
print('(Unsupervised: model learns normal pattern, flags deviations)')

history = autoencoder.fit(
    X_train_normal, X_train_normal,  # Input = Output (reconstruction)
    epochs=100,
    batch_size=256,
    validation_split=0.1,
    callbacks=callbacks,
    shuffle=True,
    verbose=1
)

print('\n✅ Autoencoder training complete')

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('DeepGuard Autoencoder — Training History', fontweight='bold')

axes[0].plot(history.history['loss'], label='Train Loss', color='#3498db')
axes[0].plot(history.history['val_loss'], label='Val Loss', color='#e74c3c')
axes[0].set_title('Training Loss Over Epochs')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Reconstruction Error Distribution
X_test_reconstructed = autoencoder.predict(X_test, verbose=0)
mse_all = np.mean(np.power(X_test.values - X_test_reconstructed, 2), axis=1)

if y_test is not None:
    mse_normal = mse_all[y_test == 0]
    mse_fraud = mse_all[y_test == 1]
    axes[1].hist(mse_normal, bins=50, alpha=0.6, color='#2ecc71',
                label=f'Legitimate (n={len(mse_normal):,})', density=True)
    axes[1].hist(mse_fraud, bins=50, alpha=0.8, color='#e74c3c',
                label=f'Laundering (n={len(mse_fraud):,})', density=True)
else:
    axes[1].hist(mse_all, bins=50, alpha=0.7, color='#3498db', density=True)

axes[1].set_title('Reconstruction Error Distribution')
axes[1].set_xlabel('Mean Squared Error')
axes[1].set_ylabel('Density')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('autoencoder_training.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Training plot saved')

In [ ]:
# ============================================================
# DETERMINE OPTIMAL THRESHOLD
# ============================================================

# Calculate MSE for validation set
X_val_reconstructed = autoencoder.predict(X_val, verbose=0)
mse_val = np.mean(np.power(X_val.values - X_val_reconstructed, 2), axis=1)

# Threshold = mean + 3 standard deviations of normal transactions
if y_val is not None:
    mse_val_normal = mse_val[y_val == 0]
    threshold = np.mean(mse_val_normal) + 3 * np.std(mse_val_normal)
else:
    threshold = np.mean(mse_val) + 3 * np.std(mse_val)

print(f'📊 MSE Threshold (mean + 3σ): {threshold:.6f}')

# Apply threshold to test set
ae_binary = (mse_all > threshold).astype(int)
# Normalize MSE to 0-100 score
ae_scores_norm = np.clip(mse_all / threshold, 0, 1)
ae_scores_pct = (ae_scores_norm * 100).round(2)

print(f'\n📊 Autoencoder Results on Test Set:')
print(f'   Anomalies detected: {ae_binary.sum():,} / {len(ae_binary):,}')

if y_test is not None:
    ae_roc = roc_auc_score(y_test, ae_scores_norm)
    ae_f1 = f1_score(y_test, ae_binary, zero_division=0)
    ae_prec = precision_score(y_test, ae_binary, zero_division=0)
    ae_rec = recall_score(y_test, ae_binary, zero_division=0)
    print(f'\n📊 Autoencoder Performance Metrics:')
    print(f'   ROC-AUC:   {ae_roc:.4f}')
    print(f'   F1 Score:  {ae_f1:.4f}')
    print(f'   Precision: {ae_prec:.4f}')
    print(f'   Recall:    {ae_rec:.4f}')

# Save threshold
model_metadata = {
    'ae_threshold': float(threshold),
    'contamination_rate': contamination_rate,
    'input_dim': input_dim,
    'feature_names': feature_names
}
with open('model_metadata.json', 'w') as f:
    json.dump(model_metadata, f, indent=2)
print('\n✅ Model metadata saved to model_metadata.json')

# Save autoencoder
autoencoder.save('autoencoder.h5')
print('✅ Autoencoder saved to autoencoder.h5')

## 🎯 Step 8: Ensemble Scoring (IF + AE Combined)

In [ ]:
print('=' * 60)
print('ENSEMBLE SCORING (50/50 Weighted Combination)')
print('=' * 60)

# Normalize IF scores to 0-1
if_norm = (if_scores_pct / 100)

# Normalize AE scores to 0-1
ae_norm = np.clip(mse_all / threshold, 0, 1)

# Ensemble: 50% IF + 50% AE
IF_WEIGHT = 0.5
AE_WEIGHT = 0.5
ensemble_scores = (IF_WEIGHT * if_norm) + (AE_WEIGHT * ae_norm)
ensemble_scores_pct = (ensemble_scores * 100).round(2)

# Binary prediction: flag if ensemble score > 50%
ensemble_binary = (ensemble_scores > 0.5).astype(int)

print(f'IF Weight: {IF_WEIGHT*100:.0f}%  |  AE Weight: {AE_WEIGHT*100:.0f}%')
print(f'Flag threshold: 50%')
print(f'Total flagged: {ensemble_binary.sum():,} / {len(ensemble_binary):,}')

if y_test is not None:
    ens_roc = roc_auc_score(y_test, ensemble_scores)
    ens_f1 = f1_score(y_test, ensemble_binary, zero_division=0)
    ens_prec = precision_score(y_test, ensemble_binary, zero_division=0)
    ens_rec = recall_score(y_test, ensemble_binary, zero_division=0)

    print(f'\n📊 ===== FINAL ENSEMBLE METRICS =====')
    print(f'   ROC-AUC:   {ens_roc:.4f} ({ens_roc*100:.1f}%)')
    print(f'   F1 Score:  {ens_f1:.4f} ({ens_f1*100:.1f}%)')
    print(f'   Precision: {ens_prec:.4f} ({ens_prec*100:.1f}%)')
    print(f'   Recall:    {ens_rec:.4f} ({ens_rec*100:.1f}%)')
    print(f'   ==================================')

    # Confusion Matrix
    cm = confusion_matrix(y_test, ensemble_binary)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Legitimate', 'Laundering'],
                yticklabels=['Legitimate', 'Laundering'])
    plt.title('DeepGuard Ensemble — Confusion Matrix', fontweight='bold')
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.tight_layout()
    plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('\n✅ Confusion matrix saved')

## 🔍 Step 9: SHAP Explainability

In [ ]:
print('=' * 60)
print('SHAP EXPLAINABILITY ANALYSIS')
print('=' * 60)

# Use small sample for SHAP computation (computationally expensive)
# Use Isolation Forest for SHAP (faster than deep model)
shap_sample_size = min(500, len(X_test))
X_shap = X_test.iloc[:shap_sample_size]

print(f'Computing SHAP values on {shap_sample_size} samples...')
print('(This may take 2-5 minutes)')

# SHAP TreeExplainer for Isolation Forest
explainer = shap.TreeExplainer(iso_forest)
shap_values = explainer.shap_values(X_shap)

print('✅ SHAP values computed')

# SHAP Summary Plot — Bar Chart
plt.figure(figsize=(12, 8))
shap.summary_plot(
    shap_values,
    X_shap,
    plot_type='bar',
    max_display=15,
    show=False
)
plt.title('DeepGuard — SHAP Feature Importance (Global)', fontweight='bold')
plt.tight_layout()
plt.savefig('shap_summary_bar.png', dpi=150, bbox_inches='tight')
plt.close()
print('✅ SHAP bar chart saved')

# SHAP Beeswarm Plot
plt.figure(figsize=(12, 8))
shap.summary_plot(
    shap_values,
    X_shap,
    max_display=15,
    show=False
)
plt.title('DeepGuard — SHAP Beeswarm Plot', fontweight='bold')
plt.tight_layout()
plt.savefig('shap_summary_beeswarm.png', dpi=150, bbox_inches='tight')
plt.close()
print('✅ SHAP beeswarm plot saved')

In [ ]:
# SHAP for individual flagged transaction
print('Generating SHAP explanation for top flagged transaction...')

# Find most suspicious transaction
top_idx = np.argmax(ensemble_scores_pct[:shap_sample_size])
top_shap = shap_values[top_idx]
top_features = X_shap.iloc[top_idx]

# Create SHAP waterfall-style bar chart for single transaction
feature_importance = dict(zip(X_shap.columns, np.abs(top_shap)))
sorted_features = sorted(feature_importance.items(), key=lambda x: x[1], reverse=True)[:10]

feat_names = [f[0][:25] for f in sorted_features]  # Truncate long names
feat_vals = [f[1] for f in sorted_features]

plt.figure(figsize=(10, 6))
colors = ['#e74c3c' if v > 0 else '#2ecc71' for v in
          [top_shap[list(X_shap.columns).index(f[0])] if f[0] in X_shap.columns else 0
           for f in sorted_features]]
bars = plt.barh(range(len(feat_names)), feat_vals[::-1], color=colors[::-1])
plt.yticks(range(len(feat_names)), feat_names[::-1])
plt.xlabel('|SHAP Value| — Feature Contribution to Fraud Score')
plt.title(f'DeepGuard SHAP — Transaction #{top_idx}\nRisk Score: {ensemble_scores_pct[top_idx]:.1f}%',
         fontweight='bold')
plt.tight_layout()
plt.savefig('shap_single_transaction.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Single transaction SHAP saved')
print(f'   Transaction Risk Score: {ensemble_scores_pct[top_idx]:.1f}%')
print(f'   Top features:')
for name, val in sorted_features[:5]:
    print(f'   - {name}: {val:.4f}')

## 📈 Step 10: Final Performance Summary

In [ ]:
if y_test is not None:
    print('=' * 60)
    print('DEEPGUARD — FINAL MODEL PERFORMANCE REPORT')
    print('=' * 60)

    results = {
        'Isolation Forest': {
            'ROC-AUC': roc_auc_score(y_test, if_scores_norm),
            'F1': f1_score(y_test, if_binary, zero_division=0),
            'Precision': precision_score(y_test, if_binary, zero_division=0),
            'Recall': recall_score(y_test, if_binary, zero_division=0)
        },
        'Autoencoder': {
            'ROC-AUC': ae_roc,
            'F1': ae_f1,
            'Precision': ae_prec,
            'Recall': ae_rec
        },
        'Ensemble (DeepGuard)': {
            'ROC-AUC': ens_roc,
            'F1': ens_f1,
            'Precision': ens_prec,
            'Recall': ens_rec
        }
    }

    for model, metrics in results.items():
        print(f'\n{model}:')
        for metric, value in metrics.items():
            bar = '█' * int(value * 20)
            print(f'  {metric:12s}: {value:.4f} ({value*100:.1f}%) {bar}')

    # Save results
    with open('model_results.json', 'w') as f:
        json.dump({
            k: {m: round(v, 4) for m, v in metrics.items()}
            for k, metrics in results.items()
        }, f, indent=2)
    print('\n✅ Results saved to model_results.json')

    # Comparison bar chart
    fig, ax = plt.subplots(figsize=(12, 6))
    metrics_names = ['ROC-AUC', 'F1', 'Precision', 'Recall']
    x = np.arange(len(metrics_names))
    width = 0.25
    colors = ['#3498db', '#e67e22', '#2ecc71']

    for i, (model, metrics) in enumerate(results.items()):
        vals = [metrics[m] for m in metrics_names]
        bars = ax.bar(x + i*width, vals, width, label=model, color=colors[i], alpha=0.85)
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
                   f'{val:.2f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

    ax.set_title('DeepGuard — Model Comparison', fontweight='bold', fontsize=13)
    ax.set_xticks(x + width)
    ax.set_xticklabels(metrics_names)
    ax.set_ylim(0, 1.1)
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('✅ Model comparison chart saved')

## 💾 Step 11: Export All Files

In [ ]:
import zipfile

print('Packaging all model files...')

files_to_zip = [
    'isolation_forest.pkl',
    'autoencoder.h5',
    'scaler.pkl',
    'model_metadata.json',
    'feature_names.json',
    'model_results.json',
    'shap_summary_bar.png',
    'shap_summary_beeswarm.png',
    'shap_single_transaction.png',
    'autoencoder_training.png',
    'model_comparison.png',
    'confusion_matrix.png',
    'eda_analysis.png'
]

with zipfile.ZipFile('deepguard_models.zip', 'w') as zf:
    for fname in files_to_zip:
        if os.path.exists(fname):
            zf.write(fname)
            print(f'  ✅ Added: {fname}')
        else:
            print(f'  ⚠️ Missing: {fname}')

print('\n🎉 All done! Downloading deepguard_models.zip...')
print('📁 Extract and place files in: fastapi/models/')

# Auto-download
from google.colab import files
files.download('deepguard_models.zip')

## ✅ Training Complete!

### Files Generated:
| File | Purpose |
|------|---------|
| `isolation_forest.pkl` | Trained Isolation Forest model |
| `autoencoder.h5` | Trained Deep Autoencoder model |
| `scaler.pkl` | Min-Max scaler for preprocessing |
| `model_metadata.json` | Threshold + config values |
| `feature_names.json` | Feature list for FastAPI |
| `shap_*.png` | SHAP explanation charts |

### Next Steps:
1. Extract `deepguard_models.zip`
2. Place all `.pkl`, `.h5`, `.json` files in `fastapi/models/`
3. Run `fastapi/main.py`
4. Run `backend/server.js`
5. Run `cd frontend && npm start`